# 02-Feature Engineering

**Project:** Crop Recommendation (Group E)  
**Objective:** Engineer scientifically justified features while retaining all original variables.


## 1) Scope

We keep all original features from the dataset:

- `N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`

We add only 3 engineered features with agronomic motivation, then test if they improve classification performance.



In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, mutual_info_classif

SEED = 42
pd.set_option("display.max_columns", 100)


In [3]:
DATA_PATH = "../data/raw/Crop_recommendation.csv"
df = pd.read_csv(DATA_PATH)

BASE_FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Classes:", df[TARGET].nunique())


Shape: (2200, 8)
Missing values: 0
Duplicate rows: 0
Classes: 22


## 2) Feature Engineering Hypotheses 
We selected these engineered features to add meaningful complexity based on agronomic reasoning, not random transformations. `moisture_index` combines rainfall and humidity to better represent effective water conditions, `P_to_K` captures nutrient balance between phosphorus and potassium rather than only absolute values, and `ph_dev` models how far soil pH is from a near-optimal zone for many crops. Together, these features test whether simple domain-informed interactions and nonlinear effects improve crop classification while still keeping all original dataset features.

### H1: Moisture Availability Index

$$
\text{moisture\_index} = \text{rainfall} \times \text{humidity}
$$

Rainfall alone does not represent effective water availability. Humidity influences evaporation and moisture retention.

### H2: Nutrient Balance Ratio

$$
\text{P\_to\_K} = \frac{P}{K}
$$

Crop response depends on nutrient balance, not only absolute nutrient amounts.

### H3: pH Deviation from Near-Optimal Zone

$$
\text{ph\_dev} = |ph - 6.5|
$$

Many crops prefer slightly acidic to neutral soil. Distance from this zone can capture nonlinear effects.



In [6]:
def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = 1e-6

    X["moisture_index"] = X["rainfall"] * X["humidity"]
    X["P_to_K"] = X["P"] / (X["K"] + eps)
    X["ph_dev"] = np.abs(X["ph"] - 6.5)

    return X


## 3) Data Quality Checks for New Features



In [7]:
X_base = df[BASE_FEATURES].copy()
y = df[TARGET].copy()

X_all = add_engineered_features(X_base)

ENGINEERED_FEATURES = ["moisture_index", "P_to_K", "ph_dev"]
FINAL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

print("Base features:", len(BASE_FEATURES))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("Total features:", len(FINAL_FEATURES))

X_all[ENGINEERED_FEATURES].describe().T


Base features: 7
Engineered features: 3
Total features: 10


,count,mean,std,min,25%,50%,75%,max
moisture_index,2200.0,7511.248120,5115.392118,939.371971,3790.851731,5806.931653,10039.004348,25321.958761
P_to_K,2200.0,1.668555,1.197217,0.090909,0.697874,1.262531,2.588235,5.999999
ph_dev,2200.0,0.589235,0.502549,0.000064,0.227128,0.475365,0.813645,3.435091


Checks performed:
- No missing values introduced
- No infinite values from division
- Feature ranges are plausible


In [8]:
print("NaN in engineered:", X_all[ENGINEERED_FEATURES].isna().any().any())
print("Inf in engineered:", np.isinf(X_all[ENGINEERED_FEATURES].to_numpy()).any())

X_all[ENGINEERED_FEATURES].corr()


NaN in engineered: False
Inf in engineered: False


,moisture_index,P_to_K,ph_dev
moisture_index,1.000000,-0.307525,-0.141114
P_to_K,-0.307525,1.000000,0.193807
ph_dev,-0.141114,0.193807,1.000000


## 4) Ablation Study: Measuring the Contribution of Each Engineered Feature

To quantify the value of each engineered variable, we evaluate models in a stepwise manner: first with the 7 original features (baseline), then by adding engineered features one at a time, and finally with all engineered features together. This isolates the performance impact of each addition.

We use **Logistic Regression** as a course-aligned baseline classifier and report **Macro-F1** under **Stratified 5-Fold Cross-Validation** to ensure robust, class-balanced evaluation.


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(X: pd.DataFrame, y: pd.Series):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=8000)),
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro")
    return scores.mean(), scores.std()

experiments = {
    "base_7_only": BASE_FEATURES,
    "base + moisture_index": BASE_FEATURES + ["moisture_index"],
    "base + P_to_K": BASE_FEATURES + ["P_to_K"],
    "base + ph_dev": BASE_FEATURES + ["ph_dev"],
    "base + all_3_engineered": FINAL_FEATURES,
}

rows = []
for name, cols in experiments.items():
    mean_f1, std_f1 = evaluate(X_all[cols], y)
    rows.append({
        "experiment": name,
        "n_features": len(cols),
        "macro_f1_mean": round(mean_f1, 4),
        "macro_f1_std": round(std_f1, 4),
    })

ablation_df = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
ablation_df


,experiment,n_features,macro_f1_mean,macro_f1_std
4,base + all_3_engineered,10,0.9772,0.0063
3,base + ph_dev,8,0.9750,0.0039
2,base + P_to_K,8,0.9713,0.0052
1,base + moisture_index,8,0.9711,0.0040
0,base_7_only,7,0.9707,0.0023


### Interpretation of Ablation Results

The baseline model with the 7 original features achieves a Macro-F1 of **0.9707**.  
Adding engineered features improves performance in all tested cases:

- `moisture_index`: **0.9711** (+0.0004)
- `P_to_K`: **0.9713** (+0.0006)
- `ph_dev`: **0.9750** (+0.0043)
- all 3 engineered features: **0.9772** (+0.0065)

Key observations:
1. **`ph_dev` is the strongest single engineered feature**, suggesting nonlinear pH effects are important for separating crop classes.
2. **`moisture_index` and `P_to_K` provide smaller but positive gains** individually.
3. **Using all three features together gives the best overall result**, indicating complementary information.

Overall, the ablation supports keeping all original features and adding the 3 engineered features for the modeling stage.


In [10]:
fs_rows = []
for k in [7, 8, 9, 10]:
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=mutual_info_classif, k=k)),
        ("model", LogisticRegression(max_iter=8000)),
    ])
    scores = cross_val_score(pipe, X_all[FINAL_FEATURES], y, cv=cv, scoring="f1_macro")
    fs_rows.append({
        "k_selected": k,
        "macro_f1_mean": round(scores.mean(), 4),
        "macro_f1_std": round(scores.std(), 4),
    })

pd.DataFrame(fs_rows)


,k_selected,macro_f1_mean,macro_f1_std
0,7,0.9589,0.0126
1,8,0.9711,0.0073
2,9,0.9717,0.0045
3,10,0.9772,0.0063


## 6) Final Decision

We retain all 7 original dataset features and include the 3 engineered features (`moisture_index`, `P_to_K`, `ph_dev`) for the final modeling stage.

The performance gain is modest but consistent (Macro-F1: 0.9707 to 0.9772), which is expected given the dataset is already clean and highly separable. We therefore treat feature engineering as a domain-informed refinement rather than a major accuracy driver.
